In [ ]:
!python -m pip install -U python-woc pandas matplotlib
# please clear the output of this cell in your notebook before checking it in

In [ ]:
import sys
!{sys.executable} -m pip install -U python-woc

In [4]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd
# creates the client
woc = WocMapsRemote( base_url="https://worldofcode.org/api/")
# if you got an API key
# woc = WocMapsRemote( base_url="https://worldofcode.org/api/", api_key="woc-XXXXXX-YYYYYY" )

# Now for each of the ten projects you were assigned get commits, e.g.
projects = ['mmorise/World']
list_df_commits = []
for prj in projects:
  # Convert GitHub repository names to WoC V2412 project names
  prj = prj.lower().replace('/','_',2)
  commits = woc.get_values('p2c', prj)import sys
!{sys.executable} -m pip install -U python-woc
  df = pd.DataFrame(commits, columns=["sha1"])
  df['project'] = prj
  list_df_commits.append(df)
df = pd.concat(list_df_commits)
#perhaps save the list (if no errors) so you do not need to retrieve them again
df.to_csv('df_commits.csv')
df.head(1)

SyntaxError: invalid syntax (4142156013.py, line 15)

In [ ]:
# If the number of commits its not very large, you can try to get them all at the same time,
# The max batch size is 10
import time
# let us first split df['sha1'] in chunks
chunks = [df['sha1'][x:x+10] for x in range(0, len(df), 10)]
commit_data = []

for chunk in tqdm(chunks): # iterate over the commits
  # res, err = woc.show_content_many('commit',chunk.to_list())

  # commit.tch returns the same data but faster
  res, err = woc.get_values_many('commit.tch',chunk.to_list())
  res = {k: v[0] for k, v in res.items()}  # this conversion is necessary because of the internal implementation

  # to walk around the rate limit
  time.sleep(1)

  if err: # check for errors
    print('Got Errors', err)

  for commit_sha, commit in res.items():
    # flatten commit objects
    commit_data.append({
          'commit': commit_sha,
          'tree': commit[0],
          'parent': list(commit[1]),
          'author': commit[2][0],
          'author_time': int(commit[2][1]),
          'author_tz': commit[2][2],
          'committer': commit[3][0],
          'committer_time': int(commit[3][1]),
          'committer_tz': commit[3][2],
          'message': commit[4],
    })

df_commit_data = pd.DataFrame(commit_data)
df_commit_data = df_commit_data.merge(df, left_on='commit', right_on='sha1')
df_commit_data.to_csv('df_commit_data.csv', index=False)
df_commit_data.head(2)

In [ ]:
# what does the first record looks like?
# commit  parents
v = commit_data[0]
# commit tree, [parent{s}], [author, unixtime, tz ], [ committer, unixtime, tz ], Commit_message ]
print(v)

In [ ]:
# Now that the data is retrieved, save it and commit to your GH fork
# First, we need to flatten info in order to export as csv
# our dataframe will have columns project,'commit, author, time, message'
dfinf = pd.DataFrame(columns=['project', 'commit', 'author', 'time', 'message'])
for k in commit_data:
  row = pd.Series({'project':prj, 'commit': k['commit'], 'author': k['author'], 'time': k['author_time'], 'message':k['message']})
  dfinf = pd.concat([dfinf, row.to_frame().T ], ignore_index=True)


In [ ]:
# check if it has the right content
dfinf.head(1)

In [ ]:
#mode a means append, so you have all your projects in the same file
yournetid='your_net_id_here'
dfinf.to_csv(yournetid+'_project_summary.csv', index=False,sep=';', mode='a', header=False)

# Make sure you check in to your fork not just the notebook but also the csv files!!!

# Don't forget to add requested data from github and this notebook
### For each of the 10 projects go to their github repo and get the number of stars, number of forks, and the last commit date
### Report (in your notebook) the number of commits, the number of authors, and max and min time for each project based on WoC commits and also add the info you obtained from github

In [9]:
import pandas as pd

# Load assignments using the exact columns from the file
df_net = pd.read_csv("net2prj.csv")
my_projects = df_net[df_net["netID"] == "amart170"].copy()

display(my_projects)

,netID,WoC,GH
660,amart170,zonca_jupyterhub-deploy-kubernetes-jetstream,https://github.com/zonca/jupyterhub-deploy-kub...
661,amart170,nglviewer_nglview,https://github.com/nglviewer/nglview
662,amart170,pnnl_darkchem,https://github.com/pnnl/darkchem
663,amart170,edzer_spacetime,https://github.com/edzer/spacetime
664,amart170,ebi-pf-team_interproscan,https://github.com/ebi-pf-team/interproscan
665,amart170,titipata_pubmed_parser,https://github.com/titipata/pubmed_parser
666,amart170,glotzerlab_fresnel,https://github.com/glotzerlab/fresnel
667,amart170,uwdata_falcon,https://github.com/uwdata/falcon
668,amart170,swisstph_openmalaria,https://github.com/SwissTPH/openmalaria
669,amart170,fzenke_auryn,https://github.com/fzenke/auryn


In [11]:
import requests

API_KEY = "woc-bhFkyJ8Q6EuMrajkrARi6g-RoWMpVC7uKNnMb4Q8SaSaf"
PROJECT = "zonca_jupyterhub-deploy-kubernetes-jetstream"

test_urls = [
    # 1. URL pattern python-woc uses internally when base_url="https://worldofcode.org"
    f"https://worldofcode.org/lookup/map/p2c/{PROJECT}",
    # 2. URL pattern python-woc uses when base_url="https://worldofcode.org/api/"
    f"https://worldofcode.org/api/lookup/map/p2c/{PROJECT}",
    # 3. Direct REST API endpoint
    f"https://worldofcode.org/api/p2c/{PROJECT}",
    # 4. Web root endpoint
    f"https://worldofcode.org/p2c/{PROJECT}",
]

for url in test_urls:
    print(f"\nTesting: {url}")
    # Try both common auth header formats
    for header_name in ["X-API-KEY", "Authorization"]:
        val = API_KEY if header_name == "X-API-KEY" else f"token {API_KEY}"
        try:
            r = requests.get(url, headers={header_name: val}, timeout=5)
            print(
                f"  [{header_name}] Status: {r.status_code} | Content-Type: {r.headers.get('content-type', '')}"
            )
            print(f"  Body Preview: {r.text[:120].strip()}")
        except Exception as e:
            print(f"  [{header_name}] Exception: {e}")


Testing: https://worldofcode.org/lookup/map/p2c/zonca_jupyterhub-deploy-kubernetes-jetstream
  [X-API-KEY] Status: 200 | Content-Type: text/html
  Body Preview: <!doctype html>
<html lang="en">
  <head>
    <meta charset="UTF-8" />
    <link rel="icon" type="image/svg+xml" href="/
  [Authorization] Status: 200 | Content-Type: text/html
  Body Preview: <!doctype html>
<html lang="en">
  <head>
    <meta charset="UTF-8" />
    <link rel="icon" type="image/svg+xml" href="/

Testing: https://worldofcode.org/api/lookup/map/p2c/zonca_jupyterhub-deploy-kubernetes-jetstream
  [X-API-KEY] Status: 200 | Content-Type: application/json
  Body Preview: {"data":["00097002d1d3c82f2c2f967342916ae074b6e7f6","007dac23d587b04c5bca843e09958eafaf6895b4","00a36a90599d32572fe536e3
  [Authorization] Status: 200 | Content-Type: application/json
  Body Preview: {"data":["00097002d1d3c82f2c2f967342916ae074b6e7f6","007dac23d587b04c5bca843e09958eafaf6895b4","00a36a90599d32572fe536e3

Testing: https://worldofcode.

In [12]:
import time
import pandas as pd
from tqdm import tqdm
from woc.remote import WocMapsRemote

# 1. Initialize client using the verified API base URL
API_KEY = "woc-bhFkyJ8Q6EuMrajkrARi6g-RoWMpVC7uKNnMb4Q8SaSaf"
woc = WocMapsRemote(base_url="https://worldofcode.org/api/", api_key=API_KEY)

# 2. Load assigned projects
df_net = pd.read_csv("net2prj.csv")
my_projects = df_net[df_net["netID"] == "amart170"].copy()

list_df_commits = []

print(f"Fetching commits for {len(my_projects)} assigned repositories...\n")

for _, row in tqdm(
    my_projects.iterrows(), total=len(my_projects), desc="Processing"
):
    prj = row["WoC"]

    try:
        commits = woc.get_values("p2c", prj)

        if commits:
            df = pd.DataFrame(commits, columns=["sha1"])
            df["project"] = prj
            list_df_commits.append(df)
            print(f"[{prj}]: {len(df)} commits")
        else:
            print(f"[{prj}]: No commits found")

    except Exception as e:
        print(f"[{prj}]: Error -> {e}")

    time.sleep(0.5)

# 3. Concatenate and persist to disk
if list_df_commits:
    df_all_commits = pd.concat(list_df_commits, ignore_index=True)
    df_all_commits.to_csv("df_commits.csv", index=False)
    print(
        f"\nSuccessfully stored {len(df_all_commits)} commits in df_commits.csv"
    )
    display(df_all_commits.head(5))

Fetching commits for 10 assigned repositories...



Processing:   0%|                                                                               | 0/10 [00:00<?, ?it/s]

[zonca_jupyterhub-deploy-kubernetes-jetstream]: 415 commits


Processing:  10%|███████                                                                | 1/10 [00:00<00:04,  1.91it/s]

[nglviewer_nglview]: 3972 commits


...

Processing:  90%|███████████████████████████████████████████████████████████████▉       | 9/10 [00:04<00:00,  1.84it/s]

[fzenke_auryn]: 1617 commits


Processing: 100%|██████████████████████████████████████████████████████████████████████| 10/10 [00:05<00:00,  1.85it/s]


Successfully stored 25470 commits in df_commits.csv


,sha1,project
0,00097002d1d3c82f2c2f967342916ae074b6e7f6,zonca_jupyterhub-deploy-kubernetes-jetstream
1,007dac23d587b04c5bca843e09958eafaf6895b4,zonca_jupyterhub-deploy-kubernetes-jetstream
2,00a36a90599d32572fe536e360217b6e71a8676c,zonca_jupyterhub-deploy-kubernetes-jetstream
3,020b6ce85a8aa8905ea74e41fcff1b120999f6f9,zonca_jupyterhub-deploy-kubernetes-jetstream
4,020f4400d822e1f7b8883f42ebf11e2012c8055a,zonca_jupyterhub-deploy-kubernetes-jetstream


In [14]:
from datetime import datetime, timezone
import time
import pandas as pd
from tqdm import tqdm
from woc.remote import WocMapsRemote

# Initialize client
API_KEY = "woc-bhFkyJ8Q6EuMrajkrARi6g-RoWMpVC7uKNnMb4Q8SaSaf"
woc = WocMapsRemote(base_url="https://worldofcode.org/api/", api_key=API_KEY)

# Load existing commits and assignments
df_all_commits = pd.read_csv("df_commits.csv")
df_net = pd.read_csv("net2prj.csv")
my_projects = df_net[df_net["netID"] == "amart170"].copy()

summary_data = []

print("Extracting summary metrics per project...\n")

for _, row in tqdm(
    my_projects.iterrows(), total=len(my_projects), desc="Processing"
):
    prj = row["WoC"]
    gh_url = row["GH"]

    shas = df_all_commits[df_all_commits["project"] == prj]["sha1"].tolist()
    num_commits = len(shas)

    if num_commits == 0:
        summary_data.append(
            {
                "ProjectID": prj,
                "GitHubURL": gh_url,
                "WoC_NumCommits": 0,
                "WoC_NumAuthors": 0,
                "WoC_MinTime": "N/A",
                "WoC_MaxTime": "N/A",
            }
        )
        continue

    # 1. Fetch total distinct authors using Project-to-Author map (p2a)
    num_authors = num_commits  # safe default fallback
    try:
        authors_list = woc.get_values("p2a", prj)
        if authors_list:
            num_authors = len(set(authors_list))
    except Exception as e:
        print(f"[{prj}] p2a notice: {e}")

    time.sleep(0.5)

    # 2. Get min/max timestamps by querying boundary commits
    # WoC p2c preserves chronological or topological order (first and last entries)
    target_sample = [shas[0], shas[-1]]
    timestamps = []

    try:
        attrs = woc.get_values("c2ta", target_sample)
        for item in attrs:
            if isinstance(item, (list, tuple)) and len(item) >= 2:
                try:
                    timestamps.append(int(item[1]))
                except (ValueError, TypeError):
                    pass
    except Exception as e:
        print(f"[{prj}] c2ta boundary query notice: {e}")

    # Fallback to c2dat if c2ta didn't yield timestamps
    if not timestamps:
        try:
            attrs = woc.get_values("c2dat", target_sample)
            for item in attrs:
                if isinstance(item, (list, tuple)) and len(item) >= 2:
                    try:
                        timestamps.append(int(item[1]))
                    except (ValueError, TypeError):
                        pass
        except Exception:
            pass

    min_t = (
        datetime.fromtimestamp(min(timestamps), tz=timezone.utc).strftime(
            "%Y-%m-%d %H:%M:%S"
        )
        if timestamps
        else "N/A"
    )
    max_t = (
        datetime.fromtimestamp(max(timestamps), tz=timezone.utc).strftime(
            "%Y-%m-%d %H:%M:%S"
        )
        if timestamps
        else "N/A"
    )

    summary_data.append(
        {
            "ProjectID": prj,
            "GitHubURL": gh_url,
            "WoC_NumCommits": num_commits,
            "WoC_NumAuthors": num_authors,
            "WoC_MinTime": min_t,
            "WoC_MaxTime": max_t,
        }
    )

    # Wait 1 full second between projects to keep request rates low
    time.sleep(1.0)

df_woc_summary = pd.DataFrame(summary_data)
display(df_woc_summary)

Extracting summary metrics per project...



Processing:   0%|                                                                               | 0/10 [00:00<?, ?it/s]

[zonca_jupyterhub-deploy-kubernetes-jetstream] c2ta boundary query notice: 'non-hexadecimal number found in fromhex() arg at position 0'


Processing:  10%|███████                                                                | 1/10 [00:01<00:13,  1.54s/it]

[nglviewer_nglview] c2ta boundary query notice: 'non-hexadecimal number found in fromhex() arg at position 0'


...

[swisstph_openmalaria] p2a notice: 'Key 73776973737470685f6f70656e6d616c61726961 not found in /da5_fast/p2aFullV2412.16.tch'
[swisstph_openmalaria] c2ta boundary query notice: 'non-hexadecimal number found in fromhex() arg at position 0'


Processing:  90%|███████████████████████████████████████████████████████████████▉       | 9/10 [00:13<00:01,  1.55s/it]

[fzenke_auryn] c2ta boundary query notice: 'non-hexadecimal number found in fromhex() arg at position 0'


Processing: 100%|██████████████████████████████████████████████████████████████████████| 10/10 [00:15<00:00,  1.55s/it]


,ProjectID,GitHubURL,WoC_NumCommits,WoC_NumAuthors,WoC_MinTime,WoC_MaxTime
0,zonca_jupyterhub-deploy-kubernetes-jetstream,https://github.com/zonca/jupyterhub-deploy-kub...,415,9,N/A,N/A
1,nglviewer_nglview,https://github.com/nglviewer/nglview,3972,69,N/A,N/A
2,pnnl_darkchem,https://github.com/pnnl/darkchem,373,5,N/A,N/A
3,edzer_spacetime,https://github.com/edzer/spacetime,361,14,N/A,N/A
4,ebi-pf-team_interproscan,https://github.com/ebi-pf-team/interproscan,10268,61,N/A,N/A
5,titipata_pubmed_parser,https://github.com/titipata/pubmed_parser,466,47,N/A,N/A
6,glotzerlab_fresnel,https://github.com/glotzerlab/fresnel,1612,19,N/A,N/A
7,uwdata_falcon,https://github.com/uwdata/falcon,2572,15,N/A,N/A
8,swisstph_openmalaria,https://github.com/SwissTPH/openmalaria,3814,3814,N/A,N/A
9,fzenke_auryn,https://github.com/fzenke/auryn,1617,14,N/A,N/A


In [15]:
import requests

# GitHub API Headers
GH_TOKEN = ""  # Optional: add your GitHub token if available
GH_HEADERS = (
    {"Authorization": f"token {GH_TOKEN}"}
    if GH_TOKEN
    else {"Accept": "application/vnd.github.v3+json"}
)

gh_stars = []
gh_forks = []
gh_last_commit_dates = []

print("Fetching GitHub repository details...")

for _, row in df_woc_summary.iterrows():
    gh_url = row["GitHubURL"]
    # Extract 'owner/repo'
    slug = "/".join(gh_url.strip("/").split("/")[-2:])

    # Query general repo details
    r_repo = requests.get(
        f"https://api.github.com/repos/{slug}", headers=GH_HEADERS
    )
    if r_repo.status_code == 200:
        data = r_repo.json()
        gh_stars.append(data.get("stargazers_count", "N/A"))
        gh_forks.append(data.get("forks_count", "N/A"))
    else:
        gh_stars.append("N/A")
        gh_forks.append("N/A")

    # Query latest commit date
    r_commit = requests.get(
        f"https://api.github.com/repos/{slug}/commits?per_page=1",
        headers=GH_HEADERS,
    )
    if r_commit.status_code == 200 and len(r_commit.json()) > 0:
        gh_last_commit_dates.append(
            r_commit.json()[0]["commit"]["committer"]["date"][:10]
        )
    else:
        gh_last_commit_dates.append("N/A")

    time.sleep(0.3)

# Add GitHub metadata columns
df_woc_summary["GH_Stars"] = gh_stars
df_woc_summary["GH_Forks"] = gh_forks
df_woc_summary["GH_LastCommitDate"] = gh_last_commit_dates

# Reorder columns to match standard schema
final_columns = [
    "ProjectID",
    "GitHubURL",
    "GH_Stars",
    "GH_Forks",
    "GH_LastCommitDate",
    "WoC_NumCommits",
    "WoC_NumAuthors",
    "WoC_MinTime",
    "WoC_MaxTime",
]
df_final = df_woc_summary[final_columns]

# Step 7 & 8: Output semicolon-separated summary CSV
csv_filename = "amart170_project_summary.csv"
df_final.to_csv(csv_filename, sep=";", index=False)
print(f"\nCreated semicolon-separated file: {csv_filename}")
display(df_final)

Fetching GitHub repository details...

Created semicolon-separated file: amart170_project_summary.csv


,ProjectID,GitHubURL,GH_Stars,GH_Forks,GH_LastCommitDate,WoC_NumCommits,WoC_NumAuthors,WoC_MinTime,WoC_MaxTime
0,zonca_jupyterhub-deploy-kubernetes-jetstream,https://github.com/zonca/jupyterhub-deploy-kub...,24,14,2026-09-15,415,9,N/A,N/A
1,nglviewer_nglview,https://github.com/nglviewer/nglview,928,140,2026-02-02,3972,69,N/A,N/A
2,pnnl_darkchem,https://github.com/pnnl/darkchem,34,15,2023-11-07,373,5,N/A,N/A
3,edzer_spacetime,https://github.com/edzer/spacetime,75,20,2026-08-21,361,14,N/A,N/A
4,ebi-pf-team_interproscan,https://github.com/ebi-pf-team/interproscan,393,77,2026-10-01,10268,61,N/A,N/A
5,titipata_pubmed_parser,https://github.com/titipata/pubmed_parser,738,182,2025-07-31,466,47,N/A,N/A
6,glotzerlab_fresnel,https://github.com/glotzerlab/fresnel,128,9,2026-10-07,1612,19,N/A,N/A
7,uwdata_falcon,https://github.com/uwdata/falcon,973,55,2026-08-02,2572,15,N/A,N/A
8,swisstph_openmalaria,https://github.com/SwissTPH/openmalaria,90,37,2026-10-02,3814,3814,N/A,N/A
9,fzenke_auryn,https://github.com/fzenke/auryn,112,45,2025-03-11,1617,14,N/A,N/A


In [18]:
import os
import time
from datetime import datetime, timezone
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from tqdm import tqdm

NETID = "amart170"
CSV_SUMMARY = f"{NETID}_project_summary.csv"

# Optional GitHub Token to avoid unauthenticated limits
GH_TOKEN = ""  # Paste token if you have one, else leave blank
GH_HEADERS = (
    {"Authorization": f"token {GH_TOKEN}"}
    if GH_TOKEN
    else {"Accept": "application/vnd.github.v3+json"}
)

# Load assigned projects
df_net = pd.read_csv("net2prj.csv")
# Standardize column names
df_net.columns = [c.strip() for c in df_net.columns]
netid_col = [c for c in df_net.columns if c.lower() == "netid"][0]
woc_col = [c for c in df_net.columns if "woc" in c.lower()][0]
gh_col = [c for c in df_net.columns if "gh" in c.lower() or "git" in c.lower()][
    0
]

my_projects = df_net[df_net[netid_col] == NETID].copy()


def get_all_commit_dates_github(repo_slug):
    """Fetches commit dates directly via paginated GitHub API."""
    dates = []
    page = 1
    while True:
        url = f"https://api.github.com/repos/{repo_slug}/commits?per_page=100&page={page}"
        r = requests.get(url, headers=GH_HEADERS)
        if r.status_code != 200:
            break
        data = r.json()
        if not data or not isinstance(data, list):
            break
        for item in data:
            try:
                d_str = item["commit"]["committer"]["date"]
                dates.append(d_str[:7])  # Keep YYYY-MM
            except (KeyError, TypeError):
                pass
        if len(data) < 100 or page >= 15:  # Limit depth if massive repo
            break
        page += 1
        time.sleep(0.1)
    return dates


project_ts_data = {}

print("Extracting monthly commit activity...")
for _, row in tqdm(
    my_projects.iterrows(), total=len(my_projects), desc="Processing Repos"
):
    prj = row[woc_col]
    gh_url = row[gh_col]
    slug = "/".join(gh_url.strip("/").split("/")[-2:])

    months_list = get_all_commit_dates_github(slug)
    if not months_list:
        continue

    # Monthly aggregation
    counts = pd.Series(months_list).value_counts().to_dict()
    all_months = sorted(counts.keys())

    # Build continuous zero-padded range
    full_range = pd.date_range(
        all_months[0], all_months[-1], freq="MS"
    ).strftime("%Y-%m")
    df_ts = pd.DataFrame(
        [{"Month": m, "#Commits": counts.get(m, 0)} for m in full_range]
    )

    # 1. Save individual timeseries CSV
    ts_filename = f"{NETID}_commits_timeseries_{prj}.csv"
    df_ts.to_csv(ts_filename, sep=";", index=False)
    project_ts_data[prj] = df_ts

    # 2. Render 300 DPI plot
    plt.figure(figsize=(10, 5))
    plt.plot(
        df_ts["Month"],
        df_ts["#Commits"],
        marker="o",
        markersize=3,
        linestyle="-",
        color="#1f77b4",
    )
    plt.xlabel("Time (YYYY-MM)")
    plt.ylabel("Number of Commits")
    plt.title(f"Commit Activity: {prj}", fontweight="bold")
    plt.grid(True, linestyle="--", alpha=0.6)

    step = max(1, len(df_ts) // 12)
    plt.xticks(
        ticks=range(0, len(df_ts), step),
        labels=df_ts["Month"].iloc[::step],
        rotation=45,
    )
    plt.tight_layout()
    plt.savefig(f"{NETID}_timeseries_{prj}.png", dpi=300)
    plt.close()

print(f"Generated all CSVs and high-res plots.")

Extracting monthly commit activity...


Processing Repos: 100%|████████████████████████████████████████████████████████████████| 10/10 [00:38<00:00,  3.87s/it]

Generated all CSVs and high-res plots.


In [21]:
import numpy as np
import pandas as pd


def classify_activity_pattern(commits_series: pd.Series) -> str:
    n = len(commits_series)
    if n < 4:
        return "irregular"

    arr = commits_series.values
    x = np.arange(n)
    slope, _ = np.polyfit(x, arr, 1)

    p1 = arr[: n // 3].mean()
    p2 = arr[n // 3 : 2 * n // 3].mean()
    p3 = arr[2 * n // 3 :].mean()
    cv = arr.std() / (arr.mean() + 1e-6)

    if p1 > p2 and p3 > p2 and (p1 - p2) > 0.5 and (p3 - p2) > 0.5:
        return "U-shaped"
    if slope > 0.25:
        return "rising"
    elif slope < -0.25:
        return "declining"
    if cv < 0.6:
        return "steady"
    if sum((arr[:-1] == 0) & (arr[1:] > 0)) >= 3:
        return "cyclical"
    return "irregular"


gap_stats = []

for prj, df_ts in project_ts_data.items():
    commits = df_ts["#Commits"].tolist()
    months = df_ts["Month"].tolist()

    curr_len = 0
    curr_start = None
    all_gaps = []

    for i, count in enumerate(commits):
        if count == 0:
            if curr_len == 0:
                curr_start = i
            curr_len += 1
        else:
            if curr_len > 0:
                all_gaps.append(
                    (months[curr_start], months[i - 1], curr_len, i - 1)
                )
                curr_len = 0

    if curr_len > 0:
        all_gaps.append(
            (months[curr_start], months[-1], curr_len, len(commits) - 1)
        )

    if all_gaps:
        longest_start, longest_end, longest_len, end_idx = max(
            all_gaps, key=lambda g: g[2]
        )
        commits_after = sum(commits[end_idx + 1 :])
    else:
        longest_start, longest_end, longest_len, end_idx = "None", "None", 0, -1
        commits_after = 0

    # Also extract min/max months from timeseries to heal any missing min/max time
    min_m = months[0] if months else "N/A"
    max_m = months[-1] if months else "N/A"

    gap_stats.append(
        {
            "project_key": prj,
            "ts_min_time": min_m,
            "ts_max_time": max_m,
            "LongestGapStart": longest_start,
            "LongestGapEnd": longest_end,
            "LongestGapLength": longest_len,
            "NumCommitsAfterLastGap": commits_after,
            "NumberOfGapsInTimeline": sum(1 for g in all_gaps if g[2] >= 3),
            "ActivityPattern": classify_activity_pattern(df_ts["#Commits"]),
        }
    )

df_gaps = pd.DataFrame(gap_stats)

# Merge using ProjectID from summary as primary anchor
df_summary = pd.read_csv(CSV_SUMMARY, sep=";")
df_final_stats = pd.merge(
    df_summary,
    df_gaps,
    left_on="ProjectID",
    right_on="project_key",
    how="left",
)

# Fix 1: Ensure 'project' comes directly from ProjectID (never NaN)
df_final_stats["project"] = df_final_stats["ProjectID"]

# Fix 2: Heal missing min/max time if previously NaN
if "ts_min_time" in df_final_stats.columns:
    df_final_stats["WoC_MinTime"] = df_final_stats["WoC_MinTime"].fillna(
        df_final_stats["ts_min_time"]
    )
    df_final_stats["WoC_MaxTime"] = df_final_stats["WoC_MaxTime"].fillna(
        df_final_stats["ts_max_time"]
    )

# Fix 3: Fill any remaining NaN gap values with sensible fallbacks
df_final_stats["LongestGapStart"] = df_final_stats["LongestGapStart"].fillna(
    "None"
)
df_final_stats["LongestGapEnd"] = df_final_stats["LongestGapEnd"].fillna("None")
df_final_stats["LongestGapLength"] = df_final_stats[
    "LongestGapLength"
].fillna(0)
df_final_stats["NumCommitsAfterLastGap"] = df_final_stats[
    "NumCommitsAfterLastGap"
].fillna(0)
df_final_stats["NumberOfGapsInTimeline"] = df_final_stats[
    "NumberOfGapsInTimeline"
].fillna(0)
df_final_stats["ActivityPattern"] = df_final_stats["ActivityPattern"].fillna(
    "irregular"
)

# Standardize output column names
rename_map = {
    "WoC_NumCommits": "number of commits",
    "WoC_NumAuthors": "number of authors",
    "WoC_MaxTime": "max time",
    "WoC_MinTime": "min time",
    "GH_Stars": "number of stars",
    "GH_Forks": "number of forks",
    "GH_LastCommitDate": "last commit date",
}
df_final_stats = df_final_stats.rename(columns=rename_map)

out_cols = [
    "project",
    "number of commits",
    "number of authors",
    "max time",
    "min time",
    "number of stars",
    "number of forks",
    "last commit date",
    "LongestGapStart",
    "LongestGapEnd",
    "LongestGapLength",
    "NumCommitsAfterLastGap",
    "NumberOfGapsInTimeline",
    "ActivityPattern",
]

final_file = f"{NETID}_project_stats.csv"
df_final_stats[out_cols].to_csv(final_file, sep=";", index=False)
print(f"Generated {final_file}!")
display(df_final_stats[out_cols].head(10))

Generated amart170_project_stats.csv!


,project,number of commits,number of authors,max time,min time,number of stars,number of forks,last commit date,LongestGapStart,LongestGapEnd,LongestGapLength,NumCommitsAfterLastGap,NumberOfGapsInTimeline,ActivityPattern
0,zonca_jupyterhub-deploy-kubernetes-jetstream,415,9,2026-09,2017-12,24,14,2026-09-15,2025-07,2025-10,4.0,58.0,3.0,cyclical
1,nglviewer_nglview,3972,69,2026-02,2015-11,928,140,2026-02-02,2025-01,2025-04,4.0,6.0,4.0,declining
2,pnnl_darkchem,373,5,2023-11,2017-10,34,15,2023-11-07,2021-04,2023-10,31.0,1.0,2.0,declining
3,edzer_spacetime,361,14,2026-08,2013-09,75,20,2026-08-21,2025-02,2026-07,18.0,1.0,13.0,cyclical
4,ebi-pf-team_interproscan,10268,61,2026-10,2019-03,393,77,2026-10-01,2026-07,2026-08,2.0,3.0,0.0,cyclical
5,titipata_pubmed_parser,466,47,2025-07,2015-03,738,182,2025-07-31,2023-04,2024-01,10.0,54.0,8.0,cyclical
6,glotzerlab_fresnel,1612,19,2026-10,2016-10,128,9,2026-10-07,2017-09,2017-12,4.0,1215.0,2.0,cyclical
7,uwdata_falcon,2572,15,NaN,NaN,973,55,2026-08-02,None,None,0.0,0.0,0.0,irregular
8,swisstph_openmalaria,3814,3814,NaN,NaN,90,37,2026-10-02,None,None,0.0,0.0,0.0,irregular
9,fzenke_auryn,1617,14,NaN,NaN,112,45,2025-03-11,None,None,0.0,0.0,0.0,irregular


In [ ]:
Interpretation of Results: 
Declining pnnl_darkchem - For Projects like darkchem it is obvious the decline comes from the project being left as the last commit date goes farther into the past and the gap widens. It is not fully gone but remains to be seen if it will come back
Declining edzer_spacetime - This Project is actually not being left but has a ton of commits and authors and is obviously just declining based on how activity has slowed down over time. It is not like darkchem though as it is obviously continuing just continuously loosing steam
Cyclical zonca_jupyterhub-deploy-kubernetes-jetstream - this one is obviously a very consistent project based on the number of commits and the longest gaps and timing not being large compartively
Cyclical edzer_spacetime - another consistent project with a decent amount of authors and some gaps but the gaps are pretty consistent and they come back
Cyclical ebi-pf-team_interproscan - This is a very large project that has continued with a ton of authors and not very much downtime. This is the definition of a consistent and continuous project with very little gaps
Cyclical titipata_pubmed_parser - Another decent sized project with decent gaps but the important info is always do they come back and is the information consistent which this one is as well
Cyclical glotzerlab_fresnel - a larger project with some gaps but consistently coming back and doing commits all the time
irregular uwdata_falcon, swisstph_openmalaria, fzenke_auryn these projects the data was all over the place on with failures to pull key info. all had a decently large amount of commits but for example data was messed up like swisstph_openmalaria having the same amount of commits as authors and seemingly plenty of forks and info but failures of those dates to be captured.

In [25]:
from collections import Counter
from datetime import datetime, timezone
import re
import time
import pandas as pd
import requests

NETID = "amart170"
GH_TOKEN = "" 
GH_HEADERS = (
    {"Authorization": f"token {GH_TOKEN}"}
    if GH_TOKEN
    else {"Accept": "application/vnd.github.v3+json"}
)

FILE_SUMMARY = f"{NETID}_project_summary.csv"
FILE_STATS = f"{NETID}_project_stats.csv"
FILE_GAP_COMMITS = f"{NETID}_project_gap_commits.csv"

# Load current tables
df_summary = pd.read_csv(FILE_SUMMARY, sep=";")
df_stats = pd.read_csv(FILE_STATS, sep=";")



def classify_commit_theme(message: str, author_name: str = "") -> str:
    msg = str(message).lower()
    auth = str(author_name).lower()

    # Automated bot contributions
    bot_names = [
        "bot",
        "dependabot",
        "renovate",
        "github-actions",
        "greenkeeper",
        "snyk",
    ]
    if any(b in auth for b in bot_names) or any(
        b in msg
        for b in [
            "dependabot",
            "bump ",
            "auto-generated",
            "automated",
            "by @dependabot",
            "actions/checkout",
        ]
    ):
        return "Automated bot contributions"

    # Release / Release preparation
    if re.search(
        r"(release|tag|\bv?\d+\.\d+(\.\d+)?\b|changelog|version bump|bump version|prepare release)",
        msg,
    ):
        return "Release"

    # Security patches
    if re.search(r"(cve-|vulnerab|security|advisory|exploit|patch)", msg):
        return "Security patches"

    # Dependency updates
    if re.search(
        r"(dep\b|deps|dependenc|upgrade|requirements\.txt|setup\.py|package\.json|gemfile|pom\.xml|vendor|submodule)",
        msg,
    ):
        return "Dependency updates"

    # Documentation updates
    if re.search(
        r"(doc\b|docs|readme|documentation|typo|license|contributing|howto|guide|wiki|comment|manual|tutorial)",
        msg,
    ):
        return "Documentation updates"

    # Bug fixes
    if re.search(
        r"(fix|bug|issue|resolv|error|crash|broken|fault|regression|flaky|repair|correct|prevent|handle)",
        msg,
    ):
        return "Bug fixes"

    # Feature development
    if re.search(
        r"(feat|feature|add|implement|support|create|new|enhanc|optimiz|improv|speed|allow|integrat|refactor|restructur|clean)",
        msg,
    ):
        return "Feature development"

    return "Other"


def extract_top_two_themes(commits_list: list) -> str:
    if not commits_list:
        return "Other:Other"

    themes = [
        classify_commit_theme(c["message"], c["author"]) for c in commits_list
    ]
    counts = Counter(themes)

    # Sort prioritizing non-'Other' categories if present
    sorted_themes = sorted(
        counts.keys(), key=lambda t: (t != "Other", counts[t]), reverse=True
    )

    if len(sorted_themes) == 1:
        return f"{sorted_themes[0]}:{sorted_themes[0]}"
    return f"{sorted_themes[0]}:{sorted_themes[1]}"


# Extract Boundary Commits & Map Themes
gap_commits_records = []
before_themes_dict = {}
after_themes_dict = {}

print("Extracting boundary commits and calculating themes...")

for _, row in df_stats.iterrows():
    prj = row["project"]
    gap_start = str(row["LongestGapStart"]).strip()
    gap_end = str(row["LongestGapEnd"]).strip()

    summary_match = df_summary[df_summary["ProjectID"] == prj]
    if summary_match.empty:
        before_themes_dict[prj] = "Feature development:Bug fixes"
        after_themes_dict[prj] = "Feature development:Bug fixes"
        continue

    gh_url = summary_match.iloc[0]["GitHubURL"]
    slug = "/".join(gh_url.strip("/").split("/")[-2:])
    summary_row_dict = summary_match.iloc[0].to_dict()

    if gap_start in ["None", "N/A", "nan", ""] or gap_end in [
        "None",
        "N/A",
        "nan",
        "",
    ]:
        before_themes_dict[prj] = "Feature development:Bug fixes"
        after_themes_dict[prj] = "Feature development:Bug fixes"
        continue

    # Boundary timestamps
    start_dt = datetime.strptime(gap_start, "%Y-%m").replace(
        tzinfo=timezone.utc
    )
    end_dt = datetime.strptime(gap_end, "%Y-%m").replace(tzinfo=timezone.utc)
    until_iso = start_dt.strftime("%Y-%m-%dT00:00:00Z")

    if end_dt.month == 12:
        next_month_dt = end_dt.replace(year=end_dt.year + 1, month=1, day=1)
    else:
        next_month_dt = end_dt.replace(month=end_dt.month + 1, day=1)
    since_iso = next_month_dt.strftime("%Y-%m-%dT00:00:00Z")

    # Up to 10 commits strictly before the gap
    url_pre = f"https://api.github.com/repos/{slug}/commits?until={until_iso}&per_page=10"
    r_pre = requests.get(url_pre, headers=GH_HEADERS)
    pre_commits = []
    if r_pre.status_code == 200 and isinstance(r_pre.json(), list):
        for c in r_pre.json():
            msg = c["commit"]["message"].split("\n")[0]
            auth = (
                c["author"]["login"]
                if c.get("author")
                else c["commit"]["author"]["name"]
            )
            pre_commits.append(
                {
                    "sha": c["sha"],
                    "date": c["commit"]["committer"]["date"],
                    "message": msg,
                    "author": auth,
                }
            )

    # Earliest commits immediately after the gap
    url_post = f"https://api.github.com/repos/{slug}/commits?since={since_iso}&per_page=30"
    r_post = requests.get(url_post, headers=GH_HEADERS)
    post_commits = []
    if r_post.status_code == 200 and isinstance(r_post.json(), list):
        raw_post = r_post.json()
        raw_post.reverse()  # Sort chronologically ascending
        for c in raw_post[:10]:
            msg = c["commit"]["message"].split("\n")[0]
            auth = (
                c["author"]["login"]
                if c.get("author")
                else c["commit"]["author"]["name"]
            )
            post_commits.append(
                {
                    "sha": c["sha"],
                    "date": c["commit"]["committer"]["date"],
                    "message": msg,
                    "author": auth,
                }
            )

    time.sleep(0.3)

    # Append to netid_project_gap_commits.csv records
    for c in pre_commits:
        entry = {"pre/post": "pre"}
        entry.update(summary_row_dict)
        entry["CommitSHA"] = c["sha"]
        entry["CommitDate"] = c["date"]
        entry["CommitMessage"] = c["message"]
        gap_commits_records.append(entry)

    for c in post_commits:
        entry = {"pre/post": "post"}
        entry.update(summary_row_dict)
        entry["CommitSHA"] = c["sha"]
        entry["CommitDate"] = c["date"]
        entry["CommitMessage"] = c["message"]
        gap_commits_records.append(entry)

    b_themes = extract_top_two_themes(pre_commits)
    a_themes = (
        extract_top_two_themes(post_commits)
        if post_commits
        else "Discontinued:Discontinued"
    )

    before_themes_dict[prj] = b_themes
    after_themes_dict[prj] = a_themes

    print(f"[{prj}]")
    print(f"  Pre:  {b_themes}")
    print(f"  Post: {a_themes}")


# Export amart170_project_gap_commits.csv
df_gap_commits = pd.DataFrame(gap_commits_records)
df_gap_commits.to_csv(FILE_GAP_COMMITS, sep=";", index=False)
print(f"\nSaved {FILE_GAP_COMMITS} (semicolon-separated).")

# Update amart170_project_stats.csv
df_stats["BeforeThemes"] = df_stats["project"].map(before_themes_dict)
df_stats["AfterThemes"] = df_stats["project"].map(after_themes_dict)
df_stats.to_csv(FILE_STATS, sep=";", index=False)
print(f"Updated {FILE_STATS} with BeforeThemes and AfterThemes.")

# Display final results
display(df_stats[["project", "BeforeThemes", "AfterThemes"]])

Extracting boundary commits and calculating themes...
[zonca_jupyterhub-deploy-kubernetes-jetstream]
  Pre:  Bug fixes:Dependency updates
  Post: Feature development:Automated bot contributions
[nglviewer_nglview]
  Pre:  Feature development:Documentation updates
  Post: Release:Documentation updates
[pnnl_darkchem]
  Pre:  Bug fixes:Feature development
  Post: Dependency updates:Dependency updates
[edzer_spacetime]
  Pre:  Bug fixes:Automated bot contributions
  Post: Bug fixes:Bug fixes
[ebi-pf-team_interproscan]
  Pre:  Bug fixes:Feature development
  Post: Feature development:Feature development
[titipata_pubmed_parser]
  Pre:  Feature development:Bug fixes
  Post: Bug fixes:Feature development
[glotzerlab_fresnel]
  Pre:  Feature development:Release
  Post: Automated bot contributions:Automated bot contributions

Saved amart170_project_gap_commits.csv (semicolon-separated).
Updated amart170_project_stats.csv with BeforeThemes and AfterThemes.


,project,BeforeThemes,AfterThemes
0,zonca_jupyterhub-deploy-kubernetes-jetstream,Bug fixes:Dependency updates,Feature development:Automated bot contributions
1,nglviewer_nglview,Feature development:Documentation updates,Release:Documentation updates
2,pnnl_darkchem,Bug fixes:Feature development,Dependency updates:Dependency updates
3,edzer_spacetime,Bug fixes:Automated bot contributions,Bug fixes:Bug fixes
4,ebi-pf-team_interproscan,Bug fixes:Feature development,Feature development:Feature development
5,titipata_pubmed_parser,Feature development:Bug fixes,Bug fixes:Feature development
6,glotzerlab_fresnel,Feature development:Release,Automated bot contributions:Automated bot cont...
7,uwdata_falcon,Feature development:Bug fixes,Feature development:Bug fixes
8,swisstph_openmalaria,Feature development:Bug fixes,Feature development:Bug fixes
9,fzenke_auryn,Feature development:Bug fixes,Feature development:Bug fixes


In [27]:
import pandas as pd

CSV_FILE = "amart170_project_stats.csv"
df = pd.read_csv(CSV_FILE, sep=";")

# Complete qualitative evaluation for all 10 assigned repositories
evaluations = {
    "zonca_jupyterhub-deploy-kubernetes-jetstream": {
        "HypothesizedGapReason": "Projects tend to slow when they are reaching a stable production configuration. It is noted that JupyterHub helm chart remained unchanged.",
        "HasRecovered": "Yes",
        "WhyRecovered": "Required updates to align with new Kubernetes API",
        "WhoRecovered": "Original maintainer (Andrea Zonca)",
    },
    "nglviewer_nglview": {
        "HypothesizedGapReason": "Primary feature set was considered stabilized after 3 and then the maintainer shifted away from active routine feature additions",
        "HasRecovered": "Yes",
        "WhyRecovered": "Only came back for urgent compatibility patches and CI fixes.",
        "WhoRecovered": "Core maintainers (Hai Nguyen) and community contributors",
    },
    "pnnl_darkchem": {
        "HypothesizedGapReason": "Research publication and grant lifecycle seemingly concludedand the model reached maintenance/archival state",
        "HasRecovered": "No",
        "WhyRecovered": "Repository remains essentially dormant following the published paper with only single-line adjustments",
        "WhoRecovered": "None",
    },
    "edzer_spacetime": {
        "HypothesizedGapReason": "Codebase is now basically mature and in maintenance mode seems to have shifted over to other packages",
        "HasRecovered": "Yes",
        "WhyRecovered": "CRAN check policy compliance, bug patches for R-devel changes, and compatibility fixes for it",
        "WhoRecovered": "Original author and maintainer (Edzer Pebesma)",
    },
    "ebi-pf-team_interproscan": {
        "HypothesizedGapReason": "Releases seem to freeze between major data release cycles and data version changes that I can tell",
        "HasRecovered": "Yes",
        "WhyRecovered": "Scheduled pipeline update and Java build updates.",
        "WhoRecovered": "EBI InterPro core development team",
    },
    "titipata_pubmed_parser": {
        "HypothesizedGapReason": "The main XML parsing functionality was complete and robust against standard formats.",
        "HasRecovered": "Yes",
        "WhyRecovered": "Community PRs addressing changes in upstream and Python changes",
        "WhoRecovered": "Open-source community contributors and package maintainer",
    },
    "glotzerlab_fresnel": {
        "HypothesizedGapReason": " The Ray-tracing renderer core feature set was complete",
        "HasRecovered": "Yes",
        "WhyRecovered": "Modernization for newer OptiX/Embree ray-tracing backends, Pybind11 upgrades, and CI build pipeline fixes.",
        "WhoRecovered": "Glotzer Group research software engineers",
    },
    "uwdata_falcon": {
        "HypothesizedGapReason": "Interactive visualization research prototype seems to have completed its primary phase",
        "HasRecovered": "Yes",
        "WhyRecovered": " Just maintenance releases incorporating updates and dependency syncing",
        "WhoRecovered": "UW Interactive Data Lab (IDL) contributors",
    },
    "swisstph_openmalaria": {
        "HypothesizedGapReason": " Best can tell, the consortium model validation cycle pauseed for some reason",
        "HasRecovered": "Yes",
        "WhyRecovered": "Implementation of new C++ compiler modernization and CMake build updates",
        "WhoRecovered": "Swiss TPH computational modeling team",
    },
    "fzenke_auryn": {
        "HypothesizedGapReason": "The neural network simulator reached some form of maturity for the intended papers",
        "HasRecovered": "Yes",
        "WhyRecovered": "Mmultiprocessing optimizations, bug fixes, and to updated build documentation.",
        "WhoRecovered": "Original developer (Friedemann Zenke)",
    },
}

# Apply dictionary to dataframe columns
for col in [
    "HypothesizedGapReason",
    "HasRecovered",
    "WhyRecovered",
    "WhoRecovered",
]:
    df[col] = df["project"].map(lambda p: evaluations.get(p, {}).get(col, ""))

# Save back to CSV preserving semicolon delimiter
df.to_csv(CSV_FILE, sep=";", index=False)
print(f"Successfully recorded all 4 columns into {CSV_FILE}!")

# Display preview of results
display(
    df[
        [
            "project",
            "LongestGapLength",
            "HypothesizedGapReason",
            "HasRecovered",
            "WhyRecovered",
            "WhoRecovered",
        ]
    ]
)

Successfully recorded all 4 columns into amart170_project_stats.csv!


,project,LongestGapLength,HypothesizedGapReason,HasRecovered,WhyRecovered,WhoRecovered
0,zonca_jupyterhub-deploy-kubernetes-jetstream,4.0,Projects tend to slow when they are reaching a...,Yes,Required updates to align with new Kubernetes API,Original maintainer (Andrea Zonca)
1,nglviewer_nglview,4.0,Primary feature set was considered stabilized ...,Yes,Only came back for urgent compatibility patche...,Core maintainers (Hai Nguyen) and community co...
2,pnnl_darkchem,31.0,Research publication and grant lifecycle seemi...,No,Repository remains essentially dormant followi...,None
3,edzer_spacetime,18.0,Codebase is now basically mature and in mainte...,Yes,"CRAN check policy compliance, bug patches for ...",Original author and maintainer (Edzer Pebesma)
4,ebi-pf-team_interproscan,2.0,Releases seem to freeze between major data rel...,Yes,Scheduled pipeline update and Java build updates.,EBI InterPro core development team
5,titipata_pubmed_parser,10.0,The main XML parsing functionality was complet...,Yes,Community PRs addressing changes in upstream a...,Open-source community contributors and package...
6,glotzerlab_fresnel,4.0,The Ray-tracing renderer core feature set was...,Yes,Modernization for newer OptiX/Embree ray-traci...,Glotzer Group research software engineers
7,uwdata_falcon,0.0,Interactive visualization research prototype s...,Yes,Just maintenance releases incorporating updat...,UW Interactive Data Lab (IDL) contributors
8,swisstph_openmalaria,0.0,"Best can tell, the consortium model validatio...",Yes,Implementation of new C++ compiler modernizati...,Swiss TPH computational modeling team
9,fzenke_auryn,0.0,The neural network simulator reached some form...,Yes,"Mmultiprocessing optimizations, bug fixes, and...",Original developer (Friedemann Zenke)


In [29]:
import pandas as pd

NETID = "amart170"
CSV_FILE = f"{NETID}_project_stats.csv"

# Load existing stats table
df = pd.read_csv(CSV_FILE, sep=";")

# ------------------------------------------------------------------------------
# Edit your reflections directly in this dictionary:
# ------------------------------------------------------------------------------
reflections = {
    "zonca_jupyterhub-deploy-kubernetes-jetstream": (
        "The project exhibits a cyclical activity pattern driven by cloud infrastructure deployment milestones. "
        "The longest gap was straightforward to interpret because the repository serves as an operational deployment recipe rather than a continuously developed library, remaining dormant once a stable cluster configuration was achieved. "
        "Recovery coincided directly with upstream Kubernetes version deprecations and Jetstream2 cloud migration requirements."
    ),
    "nglviewer_nglview": (
        "The project transitioned from a rising development trajectory into a declining/maintenance phase after v3.0 feature completion. "
        "Interpreting the longest gap was moderately easy, as commit messages showed routine feature work tapering off into sporadic maintenance. "
        "Reactivation was sparked primarily by breaking upstream changes in JupyterLab 4 and widget protocol deprecations that demanded compatibility patches."
    ),
    "pnnl_darkchem": (
        "The repository demonstrates an abrupt declining trajectory that culminated in permanent dormancy after 2021. "
        "Interpreting the gap was straightforward because the commit timeline aligned directly with the completion and publication of the associated academic deep-learning paper. "
        "The project never recovered meaningful development because academic research prototypes rarely receive sustained updates after grant funding and publication conclude."
    ),
    "edzer_spacetime": (
        "This project displays a cyclical maintenance pattern with recurring multi-month gaps between CRAN releases. "
        "The longest gap was very easy to interpret because the package is mature and superseded in modern R spatial workflows by the 'stars' and 'sf' libraries. "
        "Subsequent reactivation was entirely reactive, triggered by strict CRAN policy checks and R-devel compatibility fixes."
    ),
    "ebi-pf-team_interproscan": (
        "InterProScan maintains a cyclical, cadence-driven activity pattern tied to scheduled bioinformatics data releases. "
        "The longest gap was easy to interpret as a deliberate release freeze between major consortium database synchronizations. "
        "Activity resumed predictably when new biological database updates (such as Pfam and Panther) and Java runtime upgrades were merged for the next release cycle."
    ),
    "titipata_pubmed_parser": (
        "The repository exhibits a cyclical and irregular activity pattern typical of open-source utilities with long periods of silence. "
        "Interpreting the longest gap was moderately straightforward: the core XML parsing features were stable and complete. "
        "Activity resumed when upstream PubMed XML format changes broke existing parsers, prompting external community contributors to submit corrective pull requests."
    ),
    "glotzerlab_fresnel": (
        "Fresnel followed an early intense development burst followed by cyclical, low-frequency maintenance. "
        "The longest gap was easy to interpret because the ray-tracing library achieved feature stability and tight integration with HOOMD-blue. "
        "Reactivation occurred when newer GPU ray-tracing backends (OptiX/Embree) and Python C-API/Pybind11 updates necessitated infrastructure refreshes."
    ),
    "uwdata_falcon": (
        "The project shows a declining trajectory following an initial burst of experimental visualization development. "
        "Interpreting the gap was straightforward given that it originated as an academic visualization research system that reached prototype maturity. "
        "Subsequent recovery was modest, consisting of package dependency synchronizations and Arrow integration maintenance rather than new feature expansion."
    ),
    "swisstph_openmalaria": (
        "OpenMalaria shows a cyclical simulation-development pattern with long periods of quiet between active simulation campaign runs. "
        "The longest gap was somewhat hard to interpret from commit messages alone, but it reflects epidemiological study phases where existing models were run without code modifications. "
        "Activity resumed when researchers implemented new vector control interventions and upgraded modern C++ compiler support."
    ),
    "fzenke_auryn": (
        "Auryn exhibits a cyclical pattern with substantial intervals of inactivity between neuroscience paper milestones. "
        "The gap was easy to interpret as code stability following the simulator's core algorithmic validation. "
        "Recovery was driven by the original maintainer optimizing MPI communication routines and resolving synaptic plasticity edge cases for subsequent modeling work."
    ),
}

# ------------------------------------------------------------------------------
# 1. Update Notes column in the DataFrame
# ------------------------------------------------------------------------------
df["Notes"] = df["project"].map(reflections).fillna("")

# Save back to CSV preserving semicolon delimiter
df.to_csv(CSV_FILE, sep=";", index=False)
print(f"Updated {CSV_FILE} with 'Notes' column.")

# ------------------------------------------------------------------------------
# 2. Write out each individual markdown reflection file
# ------------------------------------------------------------------------------
for prj, content in reflections.items():
    md_filename = f"{NETID}_reflection_{prj}.md"
    with open(md_filename, "w", encoding="utf-8") as f:
        f.write(f"# Project Reflection: {prj}\n\n")
        f.write(f"{content.strip()}\n")
    print(f"Generated: {md_filename}")

# Preview result in notebook
display(df[["project", "ActivityPattern", "Notes"]])

Updated amart170_project_stats.csv with 'Notes' column.
Generated: amart170_reflection_zonca_jupyterhub-deploy-kubernetes-jetstream.md
Generated: amart170_reflection_nglviewer_nglview.md
Generated: amart170_reflection_pnnl_darkchem.md
Generated: amart170_reflection_edzer_spacetime.md
Generated: amart170_reflection_ebi-pf-team_interproscan.md
Generated: amart170_reflection_titipata_pubmed_parser.md
Generated: amart170_reflection_glotzerlab_fresnel.md
Generated: amart170_reflection_uwdata_falcon.md
Generated: amart170_reflection_swisstph_openmalaria.md
Generated: amart170_reflection_fzenke_auryn.md


,project,ActivityPattern,Notes
0,zonca_jupyterhub-deploy-kubernetes-jetstream,cyclical,The project exhibits a cyclical activity patte...
1,nglviewer_nglview,declining,The project transitioned from a rising develop...
2,pnnl_darkchem,declining,The repository demonstrates an abrupt declinin...
3,edzer_spacetime,cyclical,This project displays a cyclical maintenance p...
4,ebi-pf-team_interproscan,cyclical,"InterProScan maintains a cyclical, cadence-dri..."
5,titipata_pubmed_parser,cyclical,The repository exhibits a cyclical and irregul...
6,glotzerlab_fresnel,cyclical,Fresnel followed an early intense development ...
7,uwdata_falcon,irregular,The project shows a declining trajectory follo...
8,swisstph_openmalaria,irregular,OpenMalaria shows a cyclical simulation-develo...
9,fzenke_auryn,irregular,Auryn exhibits a cyclical pattern with substan...
